In [1]:
from EEGStream import EEGNet
from Metrics import eval_metrics, load_seed
from Dataloader_EF import read_ef_train_si as si_data

import torch
import os
from sklearn.metrics import precision_score, recall_score, f1_score, cohen_kappa_score


class Trainer:
	def __init__(self, config):
		self.config = config
		self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

		self.model = EEGNet(num_class=2, DHRConv_width=4000, DWConv_height=30, num_DHRConv=4, num_DWConv=8).to(self.device)
		os.makedirs('Results', exist_ok=True)
	
	def z_score_um(self, train_dataset, test_dataset, eps=1e-6):
		train_data = torch.stack([train_dataset[i][0] for i in range(len(train_dataset))])
		train_mean = train_data.mean(dim=(0,2), keepdim=True)
		train_std = train_data.std(dim=(0,2),keepdim=True)
		train_data = (train_data - train_mean) / (train_std + eps)
		train_labels = torch.stack([train_dataset[i][1] for i in range(len(train_dataset))])
		train_dataset = torch.utils.data.TensorDataset(train_data, train_labels)

		eval_data = torch.stack([test_dataset[i][0] for i in range(len(test_dataset))])
		eval_data = (eval_data - train_mean) / (train_std + eps)
		eval_labels = torch.stack([test_dataset[i][1] for i in range(len(test_dataset))])
		test_dataset = torch.utils.data.TensorDataset(eval_data, eval_labels)

		return train_dataset, test_dataset
	
	def evaluate_epoch(self, exp_name, subject, eval_loader):
		self.model.load_state_dict(torch.load(exp_name + f'{subject}.pt'))
		self.model.eval()
		total_loss, total_correct = 0, 0
		all_preds, all_labels = [], []
		
		with torch.no_grad():
			for eval_nirs, eval_labels in eval_loader:
				eval_nirs = eval_nirs.to(self.device, dtype=torch.float32)
				eval_labels = eval_labels.to(self.device)
				
				outputs = self.model(eval_nirs)				
				preds = torch.argmax(outputs, dim=1)
				total_correct += (preds == eval_labels).sum().item()
				
				all_preds.extend(preds.cpu().numpy())
				all_labels.extend(eval_labels.cpu().numpy())
		
		acc = total_correct / len(eval_loader.dataset)
		precision = precision_score(all_labels, all_preds, zero_division=0)
		recall = recall_score(all_labels, all_preds, zero_division=0)
		f1 = f1_score(all_labels, all_preds, zero_division=0)
		kappa = cohen_kappa_score(all_labels, all_preds)
		
		return acc, precision, recall, f1, kappa
	
	def test_subject(self, exp_name, subject, mode):
		train_dataset, eval_dataset = si_data(subject, mode)
		if config['z_score']:
			train_dataset, eval_dataset = self.z_score_um(train_dataset, eval_dataset)
		eval_loader = torch.utils.data.DataLoader(eval_dataset, batch_size=self.config['batch_size'], shuffle=False)
		eval_acc, precision, recall, f1, kappa = self.evaluate_epoch(exp_name, subject, eval_loader)
		print(f"Subject {subject} | Eval Acc: {eval_acc:.2f}")

		return eval_acc, precision, recall, f1, kappa

# Hyperparameters
config = {
	'num_classes': 2,
	'mi_root': '../../Dataset/EF-MI-MA/MI/',
	'ma_root': '../../Dataset/EF-MI-MA/MA/',
	'wg_root' : '../../Dataset/EF-WG/WG/',
	'batch_size': 64,
	'z_score': True,
}

# Initialize and run trainer
for run in range(1):
	exp_name, mode, seeds = load_seed()
	num_subject = (29 if mode == 0 or mode == 1 else 26) # MI = 0, MA = 1, WG = 2
	acc_list, precision_list, recall_list, f1_list, kappa_list = [], [], [], [], []
	for subject in range(1, num_subject+1):
		seed = seeds[subject-1]
		# seed = random.randint(1, 2025)
		torch.manual_seed(seed)
		torch.cuda.manual_seed_all(seed)
		print(f"\n=== Subject {subject} ===")
		print(f'Seed is {seed}\n')
		trainer = Trainer(config)
		eval_acc, precision, recall, f1, kappa = trainer.test_subject(exp_name, subject, mode)
		acc_list.append(eval_acc)
		precision_list.append(precision)
		recall_list.append(recall)
		f1_list.append(f1)
		kappa_list.append(kappa)
	
	eval_metrics(exp_name, acc_list, precision_list, recall_list, f1_list, kappa_list)

/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(



=== Subject 1 ===
Seed is 1

1 Training Set EEG torch.Size([1680, 1, 30, 4000]) Labels torch.Size([1680])
1 Testing Set EEG torch.Size([60, 1, 30, 4000]) Labels torch.Size([60])
Subject 1 | Eval Acc: 0.85

=== Subject 2 ===
Seed is 42

2 Training Set EEG torch.Size([1680, 1, 30, 4000]) Labels torch.Size([1680])
2 Testing Set EEG torch.Size([60, 1, 30, 4000]) Labels torch.Size([60])
Subject 2 | Eval Acc: 0.90

=== Subject 3 ===
Seed is 42

3 Training Set EEG torch.Size([1680, 1, 30, 4000]) Labels torch.Size([1680])
3 Testing Set EEG torch.Size([60, 1, 30, 4000]) Labels torch.Size([60])
Subject 3 | Eval Acc: 0.78

=== Subject 4 ===
Seed is 42

4 Training Set EEG torch.Size([1680, 1, 30, 4000]) Labels torch.Size([1680])
4 Testing Set EEG torch.Size([60, 1, 30, 4000]) Labels torch.Size([60])
Subject 4 | Eval Acc: 0.75

=== Subject 5 ===
Seed is 42

5 Training Set EEG torch.Size([1680, 1, 30, 4000]) Labels torch.Size([1680])
5 Testing Set EEG torch.Size([60, 1, 30, 4000]) Labels torch.Size